In [1]:
!pip install qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 56.5 MB/s eta 0:00:00


In [2]:
import os
import json
import torch
import gc
import shutil
from pathlib import Path
from PIL import Image
from datasets import load_from_disk, Dataset, concatenate_datasets
from transformers import (
    Qwen2VLForConditionalGeneration, 
    AutoProcessor, 
    TrainingArguments, 
    Trainer
)
from peft import LoraConfig, get_peft_model
from qwen_vl_utils import process_vision_info
import pyarrow as pa

In [3]:


# ==========================================
# 1. KHAI BÁO ĐƯỜNG DẪN & CẤU HÌNH
# ==========================================
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
ORIGINAL_VI_CHART_PATH = "/kaggle/input/datasets/maituananh511/dataset-chart-vqa/vi_chart_dataset"
WORKING_VI_CHART_PATH = "/kaggle/working/vi_chart_dataset_tmp"

VIETNAMESE_DATA_PATH = Path("/kaggle/input/datasets/maituananh511/data-vietnamese/Data Vietnamese")
VIETNAMESE_IMAGES_PATH = VIETNAMESE_DATA_PATH / "images"
VIETNAMESE_JSONL_PATH = VIETNAMESE_DATA_PATH / "viet_chart_vqa.jsonl"

OUTPUT_DIR = "/kaggle/working/qwen2_vl_lora_viet_chart"
MERGED_DIR = "/kaggle/working/qwen2_vl_merged"


In [4]:
# Copy dataset sang working để tránh lỗi Read-only
if not os.path.exists(WORKING_VI_CHART_PATH):
    shutil.copytree(ORIGINAL_VI_CHART_PATH, WORKING_VI_CHART_PATH)

In [5]:


# ==========================================
# 2. SỬ DỤNG HÀM ALIGN SCHEMA CỦA BẠN ĐỂ GỘP DATA
# ==========================================
def normalize_turn(turn):
    if isinstance(turn, str): return {'role': 'assistant', 'content': turn}
    if isinstance(turn, dict):
        role = turn.get('role') or turn.get('from', '')
        if role in ('human', 'user'): role = 'user'
        elif role in ('gpt', 'assistant'): role = 'assistant'
        for rk in ('assistant', 'user', 'human', 'gpt'):
            if rk in turn and 'content' not in turn and 'role' not in turn:
                role = 'assistant' if rk in ('assistant', 'gpt') else 'user'
                return {'role': role, 'content': str(turn[rk])}
        content = str(turn.get('content') or turn.get('value', ''))
        return {'role': role, 'content': content}
    return {'role': 'assistant', 'content': str(turn)}

# HÀM XỬ LÝ LỖI SCHEMA :
def align_conversations_schema(dataset, reference_dataset):
    ref_conv_type = reference_dataset.data.schema.field('conversations').type
    ref_struct_type = ref_conv_type.value_type
    field_order = [ref_struct_type.field(i).name for i in range(ref_struct_type.num_fields)]
    pa_table = dataset.data.table
    conv_arr = pa_table.column('conversations').combine_chunks()
    struct_arr = conv_arr.values
    arrays = [struct_arr.field(f) for f in field_order]
    fields = [pa.field(f, pa.string()) for f in field_order]
    new_struct = pa.StructArray.from_arrays(arrays, fields=fields)
    new_conv = pa.ListArray.from_arrays(conv_arr.offsets, new_struct)
    idx = pa_table.schema.get_field_index('conversations')
    new_table = pa_table.set_column(idx, 'conversations', new_conv)
    return Dataset(new_table)

vietnamese_records = []
with open(VIETNAMESE_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): vietnamese_records.append(json.loads(line.strip()))

vn_rows = []
for record in vietnamese_records:
    img_path = VIETNAMESE_IMAGES_PATH / record['image']
    try:
        image = Image.open(img_path).convert('RGB')
        convs = [normalize_turn(t) for t in record['conversations']]
        pairs = [(convs[i], convs[i+1]) for i in range(0, len(convs) - 1, 2)]
        for idx, (q, a) in enumerate(pairs):
            record_id = record['id'] if len(pairs) == 1 else f"{record['id']}_q{idx}"
            vn_rows.append({'id': record_id, 'image': image, 'conversations': [q, a]})
    except Exception as e:
        continue

# Tạo Dataset tiếng Việt và căn chỉnh Schema
vi_vietnamese_train = Dataset.from_list(vn_rows[:-200])
vi_chart_dataset = load_from_disk(WORKING_VI_CHART_PATH)

vi_vietnamese_train = align_conversations_schema(vi_vietnamese_train, vi_chart_dataset['train'])

# Gộp thành công!
vi_chart_30k = vi_chart_dataset['train'].shuffle(seed=42).select(range(30000))
merged_train = concatenate_datasets([vi_chart_30k, vi_vietnamese_train])
print(f" Tổng mẫu sau gộp (Merged): {len(merged_train)}")

# ==========================================
# 3. ÉP KIỂU VỀ ĐỊNH DẠNG QWEN2-VL
# ==========================================
def map_to_qwen_format(example):
    q_content = example['conversations'][0]['content'].replace('<image>\n', '').replace('\n<image>', '').strip()
    a_content = example['conversations'][1]['content']
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q_content}]},
        {"role": "assistant", "content": [{"type": "text", "text": a_content}]}
    ]
    return {"messages": messages}



final_train_dataset = merged_train.map(map_to_qwen_format, remove_columns=['id', 'conversations']).shuffle(seed=42)


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


 Tổng mẫu sau gộp (Merged): 44671


Map:   0%|          | 0/44671 [00:00<?, ? examples/s]

In [6]:
# ==========================================
# 3. LOAD MODEL VÀ CẤU HÌNH LORA
# ==========================================
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.float16,  
    device_map="auto",          
    attn_implementation="sdpa"   
)

# Cấu hình LoRA 
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

In [7]:
# ==========================================
# 5. DATA COLLATOR & TRAIN
# ==========================================
def data_collator(examples):
    texts = [processor.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False) for ex in examples]
    images = [ex["image"] for ex in examples]
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    return batch

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, 
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=8,
    learning_rate=4e-5, 
    num_train_epochs=1, 
    
    fp16=True,            
    bf16=False,            
    
    save_steps=500, 
    logging_steps=100,
    remove_unused_columns=False, 
    optim="adamw_torch",
    dataloader_num_workers=4  
)

trainer = Trainer(model=model, args=training_args, train_dataset=final_train_dataset, data_collator=data_collator)
trainer.train()

Step,Training Loss
100,8.289999
200,6.585022
300,6.540357
400,6.470141
500,6.459377
600,6.522524
700,6.523611
800,6.558204
900,6.512693
1000,6.522141


OutOfMemoryError: CUDA out of memory. Tried to allocate 274.00 MiB. GPU 1 has a total capacity of 14.56 GiB of which 125.81 MiB is free. Including non-PyTorch memory, this process has 14.44 GiB memory in use. Of the allocated memory 13.97 GiB is allocated by PyTorch, and 337.76 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ==========================================
# 5. MERGE VÀ LƯU
# ==========================================
trainer.save_model(f"{OUTPUT_DIR}/final_adapter")
print(" Đang Merge Model...")
del model; gc.collect(); torch.cuda.empty_cache()

from peft import PeftModel
# Lưu ý: Khi merge trên CPU, dùng bfloat16 là chuẩn để giữ chất lượng
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    device_map="cpu"
)
peft_model = PeftModel.from_pretrained(base_model, f"{OUTPUT_DIR}/final_adapter")
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
processor.save_pretrained(MERGED_DIR)
print(f" Hoàn tất merge! Model tại: {MERGED_DIR}")

# ==========================================
# 6. NÉN ZIP ĐỂ TẢI VỀ
# ==========================================
import shutil
import os

# Tên file zip sẽ tạo ra: /kaggle/working/qwen2_vl_model_final.zip
zip_name = "/kaggle/working/qwen2_vl_model_final"

print(f" Đang nén model từ {MERGED_DIR}...")
# Nén toàn bộ nội dung bên trong MERGED_DIR vào file zip
shutil.make_archive(zip_name, 'zip', MERGED_DIR)

# Tính toán dung lượng file zip
zip_file = f"{zip_name}.zip"
if os.path.exists(zip_file):
    size_gb = os.path.getsize(zip_file) / (1024**3)
    print(f" Đã nén xong! File của bạn: {zip_file}")
    print(f" Dung lượng: {size_gb:.2f} GB")
else:
    print(" Lỗi: Không thể tạo file zip.")